## This is the notebook where i will try to finetune all the parameters og gemma 3 model as it has only 270 million parameter which is less as compared to 7B parameters of the other model

In [3]:
import pandas as pd
from datasets import Dataset

# Load CSV with pandas
df = pd.read_csv("/content/finetune_dataset.csv")
print(df.head())

# Converting it to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
print(dataset)



                                          input_text  \
0  Context: Since our original focus on PC graphi...   
1  Context: Some of the most recent applications ...   
2  Context: Our invention of the GPU in 1999 defi...   
3  Context: NVIDIA has a platform strategy, bring...   
4  Context: With our introduction of the CUDA pro...   

                                               label  
0           NVIDIA initially focused on PC graphics.  
1  Recent applications of GPU-powered deep learni...  
2                   NVIDIA invented the GPU in 1999.  
3  NVIDIA's platform strategy brings together har...  
4  NVIDIA's CUDA programming model opened the par...  
Dataset({
    features: ['input_text', 'label'],
    num_rows: 7000
})


In [4]:
first_row = dataset[0]  # first row as a dict for displaying
print(first_row)

{'input_text': 'Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?', 'label': 'NVIDIA initially focused on PC graphics.'}


In [10]:
from transformers import AutoTokenizer

model_name = "google/gemma-3-270m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [15]:
from datasets import load_dataset

# Load CSV
dataset = load_dataset("csv", data_files="/content/finetune_dataset.csv", split="train")

# drop rows with missing/empty fields coerce to str
def _valid(row):
    it = row.get("input_text")
    lb = row.get("label")
    return (it is not None) and (lb is not None) and (str(it).strip() != "") and (str(lb).strip() != "")

dataset = dataset.filter(_valid)
dataset = dataset.map(lambda r: {"input_text": str(r["input_text"]), "label": str(r["label"])})

len(dataset), dataset[0]


Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6998 [00:00<?, ? examples/s]

(6998,
 {'input_text': 'Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?',
  'label': 'NVIDIA initially focused on PC graphics.'})

In [ ]:
from transformers import AutoTokenizer

model_name = "google/gemma-3-270m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

MAXLEN = 512

def tokenize_for_causal_lm(batch, max_length=MAXLEN):
    # tokenize inputs
    inp = tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=max_length
    )
    # tokenize labels using text_target
    tgt = tokenizer(
        text_target=batch["label"],
        truncation=True,
        max_length=max_length
    )

    input_ids_list = []
    attn_mask_list = []
    labels_list = []

    for input_ids, attn, label_ids in zip(inp["input_ids"], inp["attention_mask"], tgt["input_ids"]):
        # masking the prompt token with -100
        labels = [-100] * len(input_ids)

        # answer to input merged
        combined_input_ids = (input_ids + label_ids)[:max_length]
        combined_attn = (attn + [1]*len(label_ids))[:max_length]

        # training with teacher forcing :: same se12seq training
        labels = (labels + label_ids)[:max_length]

        # if all truncated keep -100
        if all(l == -100 for l in labels) and len(label_ids) > 0:
            # compute loss with the last token
            labels[-1] = label_ids[-1]

        input_ids_list.append(combined_input_ids)
        attn_mask_list.append(combined_attn)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attn_mask_list,
        "labels": labels_list
    }

tokenized_dataset = dataset.map(tokenize_for_causal_lm, batched=True, remove_columns=dataset.column_names)
print(tokenized_dataset[0].keys())


Map:   0%|          | 0/6998 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [18]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhi1199 (abhi1199-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:

import os
import wandb
from huggingface_hub import login as hf_login


os.environ["WANDB_PROJECT"] = "gemma270m_full_ft"

from datasets import DatasetDict
split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
data = DatasetDict({'train': split['train'], 'validation': split['test']})

# model with its tokeniser
import torch, math
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-3-270m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

# training argument
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

# custom callback to log perplexity to W&B at every eval ## referenced from chatGPT
class LogPerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            ppl = math.exp(metrics["eval_loss"]) if metrics["eval_loss"] < 20 else float("inf")
            metrics["eval_perplexity"] = ppl
            # log explicitly to W&B as well
            try:
                wandb.log({"eval/perplexity": ppl, "eval/loss": metrics["eval_loss"]}, step=state.global_step)
            except Exception:
                pass



config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

In [23]:
args = TrainingArguments(
    output_dir="./gemma270m_full_ft",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=50,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    gradient_checkpointing=False,
    fp16=not use_bf16,
    bf16=use_bf16,
    report_to=["wandb"],
    run_name="gemma270m_full_ft",
)




In [26]:
from torch.utils.data import DataLoader
from transformers import default_data_collator


from typing import List, Dict
import torch

def collate_fn(batch: List[Dict]):
    input_batch = {
        "input_ids": [b["input_ids"] for b in batch],
        "attention_mask": [b["attention_mask"] for b in batch],
    }
    padded = tokenizer.pad(
        input_batch,
        padding="longest",
        pad_to_multiple_of=8,
        return_tensors="pt"
    )
    max_len = padded["input_ids"].shape[1]

    labels = []
    for b in batch:
        lab = b["labels"]
        if len(lab) < max_len:
            lab = lab + ([-100] * (max_len - len(lab)))
        else:
            lab = lab[:max_len]
        labels.append(lab)

    padded["labels"] = torch.tensor(labels, dtype=torch.long)
    return padded


import math
from transformers import EarlyStoppingCallback, TrainerCallback

class LogPerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            ppl = math.exp(metrics["eval_loss"]) if metrics["eval_loss"] < 20 else float("inf")
            metrics["eval_perplexity"] = ppl
            try:
                import wandb
                wandb.log({"eval/perplexity": ppl, "eval/loss": metrics["eval_loss"]}, step=state.global_step)
            except Exception:
                pass



In [27]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    data_collator=collate_fn,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        LogPerplexityCallback(),
    ],
)
train_out = trainer.train()


You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Step,Training Loss,Validation Loss,Perplexity
200,5.484700,5.653869,285.393427
400,5.328100,5.590027,267.742938
600,5.257400,5.516130,248.670928
800,5.316900,5.431888,228.580313
1000,5.057100,5.431676,228.531924
1200,4.937100,5.410744,223.798074
1400,4.947800,5.371829,215.256219
1600,4.897700,5.388539,218.883328
1800,4.885300,5.387782,218.717648
2000,4.927000,5.364395,213.661960


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:

# logging with wandb
eval_metrics = trainer.evaluate()
final_ppl = math.exp(eval_metrics["eval_loss"]) if eval_metrics["eval_loss"] < 20 else float("inf")
print("Final eval loss:", eval_metrics["eval_loss"])
print("Final perplexity:", final_ppl)
try:
    wandb.log({"final/eval_loss": eval_metrics["eval_loss"], "final/perplexity": final_ppl})
except Exception:
    pass

#save full model & tokenizer for future inference
trainer.save_model("./gemma270m_full_ft")
tokenizer.save_pretrained("./gemma270m_full_ft")

Final eval loss: 5.355658531188965
Final perplexity: 211.80340958843107


('./gemma270m_full_ft/tokenizer_config.json',
 './gemma270m_full_ft/special_tokens_map.json',
 './gemma270m_full_ft/tokenizer.model',
 './gemma270m_full_ft/added_tokens.json',
 './gemma270m_full_ft/tokenizer.json')

In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./gemma270m_full_ft"

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager"
).eval()


In [34]:
tok_len = len(tokenizer)
emb_len = model.get_input_embeddings().num_embeddings
print("tokenizer len:", tok_len, "embeddings:", emb_len)

if emb_len != tok_len:
    model.resize_token_embeddings(tok_len)

    try:
        model.tie_weights()
    except Exception:
        pass


tokenizer len: 262145 embeddings: 262144


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [38]:
def ask_model(prompt, max_new_tokens=128, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


In [39]:
prompt1 = """Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
Question: What type of organizations is the Open Value agreements designed for and what licenses does it include?
Answer:"""

In [40]:
print("Response 1:", ask_model(prompt1))

RuntimeError: probability tensor contains either `inf`, `nan` or element < 0

In [41]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./gemma270m_full_ft"

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=dtype,
    attn_implementation="eager"
).eval()

# make sure vocab size matches embeddings
tok_len = len(tokenizer)
emb_len = model.get_input_embeddings().num_embeddings
if emb_len != tok_len:
    model.resize_token_embeddings(tok_len)


In [44]:
def ask_model(prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


In [45]:
print("Greedy:", ask_model(prompt1))



The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Greedy:  licenses software licenses and- licenses software- and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and and


In [46]:
from transformers import LogitsProcessor

class NanInfGuard(LogitsProcessor):
    def __call__(self, input_ids, scores):
        return torch.nan_to_num(scores, nan=0.0, posinf=-1e9, neginf=-1e9)

def ask_model(prompt, max_new_tokens=96, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=max(temperature, 1e-5),
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=[NanInfGuard()],
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


In [47]:
print(ask_model(prompt1, temperature=0.8))


term their Purchase subscription open with Subscriptionholders annually Software years Agreement acquisitionware including season oneMicrosoft Theysubscription subscriptions availablewares programs free eitherto monthSubscriptionSoftware company onlineThe year' infreesalesually be purchased as term fromFor licensing they renew upon an agreementperiod typically allowing at certain periods various renewal months occasionally sale lease leasesIn purchases revenue Series other such like costs pay quarterly contracts Seasonstime Windows revenues time generally Additionally offers periodicase whenThey also marketing provide renewals offer value Agreements these


In [48]:
print(ask_model(prompt1, temperature=0.7))

Agreement Subscription SoftwareThe customerssually Purchase year inMicrosoft with open as six monthsperiod term subscription annualwareable lic such Seasonslett allows month' choosesale annually purchases seasonterm receive lease five leasesSA subscriptionshold whichto Opt contracts acquisition its at renewSubscriptionSoftwareOpt includes including an all premiumsubscription value allowing licensees licensing has program free allowtime sale timeLic uponUpon sales pricing Choice Period yearswares prices fees renewalparty LeaseFor more terms offers discounts two types commercial four O oneo
